In [1]:
# Connect Drive (mounts Google Drive into the Colab VM)
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os

DATA_DIR = '/content/drive/MyDrive/RFI_RAG_Project'

# Show everything in the dataset folder
for root, dirs, files in os.walk(DATA_DIR):     #Tradeoff: os.walk is just for looking while os.listdir (non-recursive, wouldn't see subfolders) or glob (processing). Chose walk here just to discover/verify what's present; this is the exploration step.
    for f in sorted(files):
        print(os.path.join(root, f).replace(DATA_DIR, '.'))

./rfi_weekly_report_v1.html
./DOCx/oac_meeting_minutes.docx
./DOCx/project_specifications.docx
./CSVs/rfi_log.csv
./PDFs/drawing_general_notes.pdf
./PDFs/field_observation_notes.pdf
./PDFs/historical_rfi_responses.pdf


In [3]:
#PDF Ingestion
!pip install pdfplumber
import os
import pdfplumber

def extract_pdf(path):
    """Read a PDF into a list of per-page records, each tagged with where it came from."""
    fname = os.path.basename(path)
    with pdfplumber.open(path) as pdf:
        for i, page in enumerate(pdf.pages, start=1): #keeps the human page number alongside each page.
            txt = page.extract_text() or "" #extract text and if it is empty or drawing return nothing
            if txt.strip(): #skip empty pages so no junk chunks.
                pages.append({
                    "text": txt,
                    "page": i,
                    "source_file": fname,
                })
    return pages



    # Testing
pages = extract_pdf(os.path.join(DATA_DIR, 'PDFs', 'historical_rfi_responses.pdf'))

print(f"{len(pages)} page-records\n")
print("FIRST RECORD:")
print("  source_file:", pages[0]["source_file"])
print("  page:       ", pages[0]["page"])
print("  text[:500]: ", pages[0]["text"][:500].replace("\n", " "))

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.0/71.0 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 76.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 92.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 93.4 MB/s eta 0:00:00
  Attempting uninstall: Pillow
    Found existing installation: pillow 11.3.0
    Uninstalling pillow-11.3.0:
      Successfully uninstalled pillow-11.3.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 5.50.0 requires pillow<12.0,>=8.0, but you have pillow 12.2.0 which is incompatible.
4 page-records

FIRST RECORD:
  source_file: historical_rfi_responses.pdf
  page:        1
  text[:500]:  RIV

In [4]:
#Docx Ingestion
!pip install python-docx
from docx import Document as Docx
import os

def extract_docx(path):
    fname = os.path.basename(path)
    segments = []
    doc = Docx(path)

    # paragraphs
    for p in doc.paragraphs:
        if p.text.strip():
            segments.append({
                "text": p.text.strip(),
                "location": "body text",
                "source_file": fname,
            })

    # tables — each row flattened to one line
    for table in doc.tables:
        for row in table.rows:
            cells = [c.text.strip() for c in row.cells if c.text.strip()]
            if cells:
                segments.append({
                    "text": " | ".join(cells),
                    "location": "table",
                    "source_file": fname,
                })

    return segments

#Testing
segs = extract_docx(os.path.join(DATA_DIR, 'DOCx', 'oac_meeting_minutes.docx'))

print(f"{len(segs)} segments\n")

# paragraph segments
print("FIRST 3 SEGMENTS:")
for s in segs[:3]:
    print(f"  [{s['location']}] {s['text'][:70]}")

# table segments
table_segs = [s for s in segs if s["location"] == "table"]   # note: "table", matching the new label
print(f"TABLE SEGMENTS FOUND: {len(table_segs)}")
for s in table_segs[:5]:
    print(f"  {s['text'][:90]}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 4.4 MB/s eta 0:00:00
21 segments

FIRST 3 SEGMENTS:
  [body text] OWNER – ARCHITECT – CONTRACTOR (OAC) MEETING MINUTES
  [body text] Riverside Medical Office Building (RMOB)  —  Meeting No. 6
  [body text] 6.1  Schedule
TABLE SEGMENTS FOUND: 3
  Date | February 24, 2026
  Location | BuildRight GC Site Trailer / Video Conference
  Attendees | Owner (Riverside Health Partners); Meridian Design Group (Architect); BuildRig


In [5]:
#Classification/Ingestion
import glob

def classify(fname):
    n = fname.lower()
    if n.endswith(".pdf")  and "historical" in n: return "historical_rfi"
    if n.endswith(".pdf")  and "drawing"    in n: return "drawing_notes"
    if n.endswith(".pdf")  and "field"      in n: return "field_notes"
    if n.endswith(".docx") and "spec"       in n: return "spec"
    if n.endswith(".docx") and "minutes"    in n: return "minutes"
    return "other"

raw_segments = []
for path in glob.glob(os.path.join(DATA_DIR, "**", "*"), recursive=True):
    fname = os.path.basename(path)
    if fname.endswith(".pdf"):
        segs = extract_pdf(path)                          # {text, page, source_file}
        for s in segs: s["location"] = s.pop("page")      # normalize page -> location
    elif fname.endswith(".docx"):
        segs = extract_docx(path)                         # {text, location, source_file}
    else:
        continue                                          # skip CSV
    dt = classify(fname)
    for s in segs:
        s["location"] = f"page {s['location']}" if isinstance(s["location"], int) else s["location"]
        s["doc_type"] = dt
    raw_segments.extend(segs)
    print(f"  {fname:34s} [{dt:14s}] → {len(segs)} segments")

print(f"\n✓ corpus ready: {len(raw_segments)} segments")

  project_specifications.docx        [spec          ] → 344 segments
  oac_meeting_minutes.docx           [minutes       ] → 21 segments
  historical_rfi_responses.pdf       [historical_rfi] → 4 segments
  drawing_general_notes.pdf          [drawing_notes ] → 1 segments
  field_observation_notes.pdf        [field_notes   ] → 1 segments

✓ corpus ready: 371 segments


In [6]:
#Chunking
!pip install -q langchain-text-splitters
import re
from langchain_text_splitters import RecursiveCharacterTextSplitter

MIN_LEN = 30

general_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=120,
    separators=["\n\n", "\n", ". ", " ", ""],   # paragraph → line → sentence → word
)

def chunk_one_segment(seg, cid_start):
    chunks, cid = [], cid_start
    for piece in general_splitter.split_text(seg["text"]):
        if len(piece.strip()) < MIN_LEN:
            continue
        chunks.append({"chunk_id": cid, "text": piece.strip(),
                       "source_file": seg["source_file"],
                       "location": seg["location"], "doc_type": seg["doc_type"]})
        cid += 1
    return chunks, cid

# ---------- rebuild all chunks ----------
chunks, cid = [], 0
for seg in raw_segments:
    new_chunks, cid = chunk_one_segment(seg, cid)
    chunks.extend(new_chunks)

print(f"{len(raw_segments)} segments → {len(chunks)} chunks\n")
for c in chunks[:3]:
    print(f"[{c['doc_type']} | {c['location']}]  {c['text'][:90]}…")

371 segments → 215 chunks

[spec | body text]  RIVERSIDE MEDICAL OFFICE BUILDING…
[spec | body text]  Volume 2 of 2 — Technical Specifications…
[spec | body text]  Architect of Record: Meridian Design Group…


In [7]:
# Testing the unstructured document here

for c in chunks:
    if c["doc_type"] == "field_notes" and "competent rock" in c["text"]:
        print("F-12 chunk:", c["chunk_id"], "| loc:", c["location"])
        print(c["text"][:240])
        break

F-12 chunk: 212 | loc: page 1
hit rock at ftg F-12, came in maybe 18in higher than where the drawings have the
bearing. everybody stood around abt an hr. called the EOR, geotech rep drove out
abt 2pm, walked it w/ me and the supt. their call: its competent rock, good
be


In [8]:
import sys
!{sys.executable} -m pip uninstall -y Pillow
!{sys.executable} -m pip install Pillow==9.5.0

#Embedding
from sentence_transformers import SentenceTransformer
import numpy as np

#local embedding model (~90MB download on first run, then cached)
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
embedder = SentenceTransformer(EMBED_MODEL)

# embed every chunk's text into a vector; stack into one matrix
chunk_texts = [c["text"] for c in chunks]
chunk_vectors = embedder.encode(
    chunk_texts,
    convert_to_numpy=True,
    normalize_embeddings=True,     # unit-length vectors -> cosine = dot product
    show_progress_bar=True,
)

print("embedded")
print("  chunks embedded:", len(chunk_texts))
print("  vector shape:   ", chunk_vectors.shape)   # (n_chunks, 384)

Found existing installation: pillow 12.2.0
Uninstalling pillow-12.2.0:
  Successfully uninstalled pillow-12.2.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 MB 11.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for Pillow: filename=Pillow-9.5.0-cp312-cp312-linux_x86_64.whl size=1210271 sha256=f2b1f2a4d032201a58044b31f5f9fd6dce019277c29b6132e1bcd212eaeba2cd
  Stored in directory: /root/.cache/pip/wheels/ea/de/2e/75a6399e5d8cd3a55c13c8f0658d996d4ce4cff37389de044c
Successfully built Pillow
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pdfplumber 0.11.10 requires Pillow>=12.2.0, but you have pillow 9.5.0 which is incompatible.
scikit-image 0.25.2 requires pillow>=10.1, but you have pillow 9.5.0 which is incompatible.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/7 [00:00<?, ?it/s]

embedded
  chunks embedded: 215
  vector shape:    (215, 384)


In [9]:
#Indexing/Storing Checkpoint
import numpy as np

# In-memory index
assert chunk_vectors.shape[0] == len(chunks), "vector/chunk count mismatch!"
norms = np.linalg.norm(chunk_vectors, axis=1)

print("index ready (in-memory)")
print("  vectors:", chunk_vectors.shape)
print("  example vector length:", round(float(norms[0]), 4), "(should be ~1.0)")

index ready (in-memory)
  vectors: (215, 384)
  example vector length: 1.0 (should be ~1.0)


In [10]:
# Load the RFI log, Open_RFIs too
import pandas as pd

df = pd.read_csv(os.path.join(DATA_DIR, "CSVs", "rfi_log.csv"))
open_rfis = df[df["Status"] == "Open"].copy()
print(f"{len(df)} total RFIs · {len(open_rfis)} open\n")
print(open_rfis[["RFI Number", "Subject"]].head(8).to_string(index=False))

28 total RFIs · 16 open

RFI Number                                             Subject
   RFI-013 Required compressive strength for level 3 slab pour
   RFI-014          Fire rating of corridor partitions level 4
   RFI-015         Sealant color at south elevation storefront
   RFI-016          Ceiling tile selection for level 2 offices
   RFI-017                Fire rating for IT room door level 3
   RFI-018           Exam room sink ADA clearance confirmation
   RFI-019     Dimension conflict grid line C - S-201 vs A-301
   RFI-020                     Unforeseen rock at footing F-12


In [11]:
# Retrieve the best chunk for each open RFI
import numpy as np

SIM_THRESHOLD = 0.57     # below this -> "no match"

def retrieve_best(question):
    q_vec = embedder.encode([question], convert_to_numpy=True, normalize_embeddings=True)
    scores = (chunk_vectors @ q_vec.T).ravel()      # cosine = dot product (normalized)
    best_i = int(np.argmax(scores))
    return best_i, float(scores[best_i])

results = []
for _, row in open_rfis.iterrows():
    best_i, score = retrieve_best(row["Question"])
    best_chunk = chunks[best_i]
    verdict = "SUGGESTED" if score >= SIM_THRESHOLD else "NO MATCH"
    results.append({
        "rfi": row["RFI Number"], "subject": row["Subject"],
        "score": round(score, 3), "verdict": verdict, "chunk": best_chunk,
    })

for r in sorted(results, key=lambda r: r["score"], reverse=True):
    tag = "✅" if r["verdict"] == "SUGGESTED" else "⛔"
    print(f"{tag} {r['rfi']}  score={r['score']:.3f}  [{r['chunk']['source_file']}]  {r['subject']}")

✅ RFI-018  score=0.808  [historical_rfi_responses.pdf]  Exam room sink ADA clearance confirmation
✅ RFI-014  score=0.752  [project_specifications.docx]  Fire rating of corridor partitions level 4
✅ RFI-020  score=0.723  [field_observation_notes.pdf]  Unforeseen rock at footing F-12
✅ RFI-017  score=0.714  [historical_rfi_responses.pdf]  Fire rating for IT room door level 3
✅ RFI-016  score=0.677  [oac_meeting_minutes.docx]  Ceiling tile selection for level 2 offices
✅ RFI-015  score=0.613  [project_specifications.docx]  Sealant color at south elevation storefront
✅ RFI-013  score=0.580  [historical_rfi_responses.pdf]  Required compressive strength for level 3 slab pour
⛔ RFI-022  score=0.537  [project_specifications.docx]  Ductwork conflict with structural beam L2
⛔ RFI-026  score=0.474  [oac_meeting_minutes.docx]  Medical gas outlet quantity exam rooms
⛔ RFI-025  score=0.396  [project_specifications.docx]  Curtain wall anchor embed missing on S-401
⛔ RFI-024  score=0.383  [project_spe

In [12]:
# the 7 RFIs we KNOW have a real answer in the documents (ground truth)
ANSWERABLE = {"RFI-013","RFI-014","RFI-015","RFI-016","RFI-017","RFI-018","RFI-020"}

def evaluate(threshold):
    tp = fp = fn = tn = 0
    for r in results:
        predicted_yes = r["score"] >= threshold
        actual_yes    = r["rfi"] in ANSWERABLE
        if   predicted_yes and actual_yes: tp += 1
        elif predicted_yes and not actual_yes: fp += 1
        elif not predicted_yes and actual_yes: fn += 1
        else: tn += 1
    precision = tp / (tp + fp) if (tp + fp) else 0
    recall    = tp / (tp + fn) if (tp + fn) else 0
    return tp, fp, fn, tn, precision, recall

print(f"{'thresh':>6} {'TP':>3} {'FP':>3} {'FN':>3} {'TN':>3} {'prec':>6} {'recall':>7}")
for t in [0.45, 0.50, 0.55, 0.58, 0.60, 0.62, 0.65]:
    tp, fp, fn, tn, p, rec = evaluate(t)
    print(f"{t:>6.2f} {tp:>3} {fp:>3} {fn:>3} {tn:>3} {p:>6.2f} {rec:>7.2f}")

thresh  TP  FP  FN  TN   prec  recall
  0.45   7   2   0   7   0.78    1.00
  0.50   7   1   0   8   0.88    1.00
  0.55   7   0   0   9   1.00    1.00
  0.58   7   0   0   9   1.00    1.00
  0.60   6   0   1   9   1.00    0.86
  0.62   5   0   2   9   1.00    0.71
  0.65   5   0   2   9   1.00    0.71


In [13]:
# ---- TIER 1: status & health (from CSV) ----
open_rfis["Days Open"]    = pd.to_numeric(open_rfis["Days Open"], errors="coerce")
open_rfis["Days Overdue"] = pd.to_numeric(open_rfis["Days Overdue"], errors="coerce")

COLS = ["RFI Number", "Subject", "Discipline", "Ball In Court", "Days Open", "Days Overdue"]

open_log     = open_rfis[COLS]
overdue_log  = open_rfis[open_rfis["Days Overdue"] > 0][COLS] \
                   .sort_values("Days Overdue", ascending=False)

# ---- TIER 2: implications (simple flag filters, no RAG) ----
cost_log     = open_rfis[open_rfis["Cost Impact"] == "Yes"][COLS]
schedule_log = open_rfis[open_rfis["Schedule Impact"] == "Yes"][COLS]

# ---- KPIs ----
kpis = {
    "open":       len(open_log),
    "overdue":    len(overdue_log),
    "answerable": sum(1 for r in results if r["verdict"] == "SUGGESTED"),
}

print("KPIs:", kpis)
print(f"\nOVERDUE ({len(overdue_log)}):")
print(overdue_log.head(5).to_string(index=False))
print(f"\nCOST-FLAGGED ({len(cost_log)}):")
print(cost_log[["RFI Number","Subject","Days Overdue"]].to_string(index=False))

KPIs: {'open': 16, 'overdue': 10, 'answerable': 7}

OVERDUE (10):
RFI Number                                             Subject    Discipline  Ball In Court  Days Open  Days Overdue
   RFI-019     Dimension conflict grid line C - S-201 vs A-301  Coordination      Architect         38            28
   RFI-016          Ceiling tile selection for level 2 offices Architectural      Architect         30            18
   RFI-013 Required compressive strength for level 3 slab pour    Structural Structural EOR         25            15
   RFI-014          Fire rating of corridor partitions level 4 Architectural      Architect         22            12
   RFI-020                     Unforeseen rock at footing F-12  Geotechnical Structural EOR         19            11

COST-FLAGGED (7):
RFI Number                                         Subject  Days Overdue
   RFI-019 Dimension conflict grid line C - S-201 vs A-301            28
   RFI-020                 Unforeseen rock at footing F-12         

In [16]:
# 3) Tier 3
import re
tier3 = []
for r in [x for x in results if x["verdict"]=="SUGGESTED"]:
    c = r["chunk"]
    txt = re.sub(r"\s+"," ", c["text"]).strip()
    tier3.append({"rfi": r["rfi"], "subject": r["subject"],
                  "answer": txt[:500] + ("…" if len(txt)>500 else ""),
                  "source": c["source_file"], "location": c["location"]})
for t in tier3:
    print(f"\n{t['rfi']} — {t['subject']}\n  {t['answer']}\n  Source: {t['source']} › {t['location']}")


RFI-013 — Required compressive strength for level 3 slab pour
  QUESTION Confirm the required 28-day compressive strength and slump for the elevated slabs on Levels 2 through 4. OFFICIAL RESPONSE Elevated slabs Levels 2-4: f'c = 4,000 psi at 28 days, maximum slump 5 in (before HRWR), normal weight, maximum water-cementitious ratio 0.45. Foundations and grade beams remain 3,000 psi. Submit mix designs for review a minimum of 10 working days prior to placement. Per Section 03 30 00 Part 2. REFERENCES S-201; Spec 03 30 00 §2.2 RFI-003 — Storefront exterior seal…
  Source: historical_rfi_responses.pdf › page 1

RFI-014 — Fire rating of corridor partitions level 4
  Corridor walls, Levels 2 through 4: 1-hour fire-resistance rated, constructed per UL Design U419 — one layer of 5/8 in. Type X gypsum board each side of 3-5/8 in. 25-gauge steel studs at 16 in. on center. Extend partitions to the underside of the structural deck and firestop all penetrations.
  Source: project_specifications.do

In [17]:
import re, os
from datetime import date

def show_answer(text, width=500):
    text = re.sub(r"\s+", " ", text).strip()
    i = text.find("OFFICIAL RESPONSE")
    if i != -1:
        text = text[i:]
    return text[:width] + ("…" if len(text) > width else "")

COLS = ["RFI Number","Subject","Discipline","Ball In Court","Days Open","Days Overdue"]

tier3 = []
for r in [x for x in results if x["verdict"] == "SUGGESTED"]:
    c = r["chunk"]
    tier3.append({"rfi": r["rfi"], "subject": r["subject"],
                  "answer": show_answer(c["text"]),
                  "source": c["source_file"], "location": c["location"]})

# building HTML report
parts = []
parts.append(f"<h1>Open RFI Weekly Report</h1>")
parts.append(f"<p>Project 2025-114 · Data as of {date(2026,6,27):%b %d, %Y}</p>")
parts.append(f"<p><b>Open:</b> {kpis['open']} &nbsp; <b>Overdue:</b> {kpis['overdue']} &nbsp; <b>Answerable from docs:</b> {kpis['answerable']}</p>")

parts.append("<h2>Tier 1 · Overdue RFIs (worst first)</h2>")
parts.append(overdue_log.to_html(index=False))

parts.append("<h2>Tier 2 · Cost implications</h2>")
parts.append(cost_log.to_html(index=False))

parts.append("<h2>Schedule implications</h2>")
parts.append(schedule_log.to_html(index=False))

parts.append("<h2>Tier 3 · Answerable from existing documents</h2>")
parts.append("<p><i>Every suggestion is advisory — verify against the source before issuing as an official response.</i></p>")
for t in tier3:
    loc = t["location"]
    loc_str = f" › {loc}" if loc not in ("body text", "table", "(document body)", "(table)") else ""
    parts.append(f"<p><b>{t['rfi']} — {t['subject']}</b><br>{t['answer']}"
                 f"<br><i>Source: {t['source']}{loc_str}</i></p>")

report = "\n".join(parts)
out = os.path.join(DATA_DIR, "rfi_weekly_report_v1.html")
with open(out, "w", encoding="utf-8") as f:
    f.write(report)
print("✓ saved to", out)

from google.colab import files
files.download(out)

✓ saved to /content/drive/MyDrive/RFI_RAG_Project/rfi_weekly_report_v1.html


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>